# Teammate #1 — Load NHANES and audit the joins

**Cycle: August 2021 - August 2023.**

**Goal:** learn which additional measurement to acquire from incomplete historical
records, accounting for uncertain relationships between measurements.

The starter notebook reuses your WDBC work. Copy functions into the marked cells
and make the listed changes. Two new functions prepare the NHANES data. The
minimax-regret engine is supplied below.

**Prediction task:** among adults aged 20+ reporting no diagnosed diabetes, predict
whether measured HbA1c is **at least 5.7%**.

**Acquisition setup:** BMI is already known. Request waist circumference or a
blood-pressure measurement, then predict using BMI and the acquired result.
Use three binary summaries: `bmi`, `waist`, and `sbp`. The blood-pressure action
reveals systolic and diastolic measurements together; this tree uses systolic BP.
Age, recorded sex and income are retained for audit. The logistic comparison
models also use age and recorded sex. Count each additional action as one unit.

**Why fewer choices than WDBC?** Two possible extra actions give four
measurement-choice rules (2 × 2). The engine also considers every assignment of
the four final prediction labels: 64 descriptions and 36 distinct predictors.
It learns predictions and acquisition choices together.

**Work order:** #1 joins tables; #2 defines and audits the cohort; #3 ports the
learners; #4 ports coverage/regret helpers and runs the experiments. Agree on column
names before working in parallel. Code marked PROVIDED is runnable. Intentionally
blank function bodies are for you to fill in your work. Clear outputs before committing.


In [1]:
from pathlib import Path
from itertools import product
import hashlib
import json
import platform
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from IPython.display import display

In [3]:
## load the required datasets
import shutil, zipfile
from pathlib import Path
import pandas as pd
from google.colab import drive

drive.mount("/content/drive",force_remount=True)

SHARED_DIR = Path("/content/drive/MyDrive/Ryan CTN Algoverse Data Drive")
print(sorted(f.name for f in SHARED_DIR.iterdir()))

Mounted at /content/drive
['DEMO_L.xpt', 'GHB_L.xpt', 'NHANES_2021-2023_all.zip', 'nhanes_summary.csv', 'nhanes_variable_list.csv', 'wdbc.data', 'wdbc.names']


In [4]:
# ---------- 1. Extract the 6 XPT files into the current folder ('.' = DATA_DIR) ----------
def _extract_nhanes_xpts(shared_dir, out_dir):
    wanted = ['DEMO_L','DIQ_L','GHB_L','BMX_L','BPXO_L','PAQ_L']
    with zipfile.ZipFile(shared_dir / "NHANES_2021-2023_all.zip") as zf:
        members = zf.namelist()
        for stem in wanted:
            hits = [m for m in members if Path(m).name.upper() == f"{stem}.XPT"]
            dest = out_dir / f"{stem}.xpt"
            if hits:
                with zf.open(hits[0]) as src, open(dest, "wb") as dst:
                    shutil.copyfileobj(src, dst)
                print(f"{stem}: extracted from zip")
            elif (shared_dir / f"{stem}.xpt").exists():
                shutil.copy(shared_dir / f"{stem}.xpt", dest)
                print(f"{stem}: copied loose file from Drive")
            else:
                print(f"{stem}: NOT FOUND. Zip contains: {members}")

_extract_nhanes_xpts(SHARED_DIR, Path('.'))
print(sorted(p.name for p in Path('.').glob('*.xpt')))

DEMO_L: extracted from zip
DIQ_L: extracted from zip
GHB_L: extracted from zip
BMX_L: extracted from zip
BPXO_L: extracted from zip
PAQ_L: extracted from zip
['BMX_L.xpt', 'BPXO_L.xpt', 'DEMO_L.xpt', 'DIQ_L.xpt', 'GHB_L.xpt', 'PAQ_L.xpt']


In [9]:
# ---------- 2. Safe XPT reader (same as the PROVIDED one) ----------
def read_local_xpt(path):
    path = Path(path)
    with path.open('rb') as handle:
        if not handle.read(80).startswith(b'HEADER RECORD'):
            raise ValueError(f'{path.name} is not an XPT file; re-download Data [XPT].')
    frame = pd.read_sas(path, format='xport')
    if 'SEQN' not in frame or frame['SEQN'].isna().any() or frame['SEQN'].duplicated().any():
        raise ValueError(f'{path.name}: participant IDs missing or duplicated.')
    return frame


In [10]:
REQUIRED_COLUMNS = {
    'DEMO_L': ['SEQN','RIDAGEYR','RIAGENDR','RIDEXPRG','INDFMPIR','SDMVSTRA','SDMVPSU'],
    'DIQ_L':  ['SEQN','DIQ010','DIQ050','DIQ070'],
    'GHB_L':  ['SEQN','LBXGH','WTPH2YR'],
    'BMX_L':  ['SEQN','BMXBMI','BMXWAIST'],
    'BPXO_L': ['SEQN','BPXOSY1','BPXOSY2','BPXOSY3','BPXODI1','BPXODI2','BPXODI3'],
    'PAQ_L':  ['SEQN','PAD680'],
}

In [11]:
def load_and_merge_nhanes(data_dir):
    data_dir = Path(data_dir)
    tables = {}
    for name, columns in REQUIRED_COLUMNS.items():
        raw = read_local_xpt(data_dir / f'{name}.xpt')
        missing = [c for c in columns if c not in raw.columns]
        if missing:
            raise ValueError(f'{name}.xpt is missing required columns: {missing}')
        table = raw[columns].copy()
        table['SEQN'] = table['SEQN'].astype('int64')
        tables[name] = table

    demo = tables['DEMO_L']
    demo_ids = set(demo['SEQN'])
    merged = demo.copy()
    merged['in_DEMO_L'] = True
    audit_rows = [{'file': 'DEMO_L', 'rows': len(demo), 'unique_ids': demo['SEQN'].nunique(),
                   'ids_matched_to_demo': len(demo), 'ids_not_in_demo': 0,
                   'columns_kept': len(REQUIRED_COLUMNS['DEMO_L']) - 1}]

    for name, table in tables.items():
        if name == 'DEMO_L':
            continue
        overlap = (set(table.columns) & set(merged.columns)) - {'SEQN'}
        if overlap:
            raise ValueError(f'{name}: columns already present in merged table: {sorted(overlap)}')
        flag = f'in_{name}'
        merged = merged.merge(table, on='SEQN', how='left',
                              validate='one_to_one', indicator=flag)
        merged[flag] = merged[flag].eq('both')
        file_ids = set(table['SEQN'])
        audit_rows.append({'file': name, 'rows': len(table), 'unique_ids': len(file_ids),
                           'ids_matched_to_demo': len(file_ids & demo_ids),
                           'ids_not_in_demo': len(file_ids - demo_ids),
                           'columns_kept': len(REQUIRED_COLUMNS[name]) - 1})

    if len(merged) != len(demo) or not merged['SEQN'].equals(demo['SEQN'].reset_index(drop=True)):
        raise AssertionError('Merged table does not contain exactly the DEMO participants.')
    if not merged['SEQN'].is_unique:
        raise AssertionError('Merged table has duplicate participant IDs.')

    join_audit = pd.DataFrame(audit_rows)
    join_audit['merged_rows'] = len(merged)
    return merged, join_audit


In [ ]:
merged, join_audit = load_and_merge_nhanes(Path('.'))
print(merged.shape)
display(join_audit)
display(merged.head())

(11933, 27)


,file,rows,unique_ids,ids_matched_to_demo,ids_not_in_demo,columns_kept,merged_rows
0,DEMO_L,11933,11933,11933,0,6,11933
1,DIQ_L,11744,11744,11744,0,3,11933
2,GHB_L,7199,7199,7199,0,2,11933
3,BMX_L,8860,8860,8860,0,2,11933
4,BPXO_L,7801,7801,7801,0,6,11933
5,PAQ_L,8153,8153,8153,0,1,11933


,SEQN,RIDAGEYR,RIAGENDR,RIDEXPRG,INDFMPIR,SDMVSTRA,SDMVPSU,in_DEMO_L,DIQ010,DIQ050,...,in_BMX_L,BPXOSY1,BPXOSY2,BPXOSY3,BPXODI1,BPXODI2,BPXODI3,in_BPXO_L,PAD680,in_PAQ_L
0,130378,43.0,1.0,NaN,5.00,173.0,2.0,True,2.0,NaN,...,True,135.0,131.0,132.0,98.0,96.0,94.0,True,360.0,True
1,130379,66.0,1.0,NaN,5.00,173.0,2.0,True,2.0,NaN,...,True,121.0,117.0,113.0,84.0,76.0,76.0,True,480.0,True
2,130380,44.0,2.0,2.0,1.41,174.0,1.0,True,1.0,2.0,...,True,111.0,112.0,104.0,79.0,80.0,76.0,True,240.0,True
3,130381,5.0,2.0,NaN,1.53,182.0,2.0,True,2.0,NaN,...,True,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False
4,130382,2.0,1.0,NaN,3.60,182.0,2.0,True,2.0,NaN,...,True,NaN,NaN,NaN,NaN,NaN,NaN,False,NaN,False


In [18]:
print(merged.columns.tolist())


['SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDEXPRG', 'INDFMPIR', 'SDMVSTRA', 'SDMVPSU', 'in_DEMO_L', 'DIQ010', 'DIQ050', 'DIQ070', 'in_DIQ_L', 'LBXGH', 'WTPH2YR', 'in_GHB_L', 'BMXBMI', 'BMXWAIST', 'in_BMX_L', 'BPXOSY1', 'BPXOSY2', 'BPXOSY3', 'BPXODI1', 'BPXODI2', 'BPXODI3', 'in_BPXO_L', 'PAD680', 'in_PAQ_L']


In [15]:
# Check that load_and_merge_nhanes followed the merge instructions.
# Prints PASS if everything is correct; otherwise stops with an AssertionError
# saying which check failed.

demo = read_local_xpt('DEMO_L.xpt')

# 1. The merged table has exactly DEMO's people, in DEMO's order,
#    so nobody was dropped or added.
assert merged['SEQN'].tolist() == demo['SEQN'].astype('int64').tolist(), 'Merged IDs do not match DEMO'

# 2. No participant ID appears twice.
assert merged['SEQN'].is_unique, 'Duplicate IDs in merged table'

# 3. No column got a _x/_y suffix (which would hide duplicate column names).
assert not any(c.endswith(('_x', '_y')) for c in merged.columns), 'Suffixed duplicate columns found'

# 4. All 6 file-presence flags (in_DEMO_L, in_DIQ_L, ...) exist.
assert all(f'in_{f}' in merged.columns for f in FILES), 'A file-presence flag is missing'

# 5. The join audit has the required columns.
assert {'file', 'rows', 'unique_ids', 'ids_matched_to_demo'} <= set(join_audit.columns), 'Join audit is missing columns'

print('PASS: merge keeps every DEMO person once, no suffixes, presence flags and audit present')


PASS: merge keeps every DEMO person once, no suffixes, presence flags and audit present


### To-be changed once all functions are created and all the teammates have run their code
```python
RUN_REAL_DATA = True # once all teammates functions are done
RUN_FULL_GRID = True # controls how big the experiment is. Is it a quick test-run or the full one. To be set to true after the quick run works
```

In [13]:
# load the required datasets

# Set this to your local/shared-Drive folder containing the six XPT files.
# Colab: mount your Drive separately if needed; add a shared-folder shortcut to My Drive.
DATA_DIR = Path('.')
OUTPUT_DIR = Path('nhanes_results')
RUN_REAL_DATA = False       # Set True after completing the functions and paths.
RUN_FULL_GRID = False       # False: 1 seed / 4 overlaps / full sample. True: 24 fits.

CONFIG = {'seeds':[0,1,2], 'sample_fractions':[0.5,1.0],
          'overlaps':[0.0,0.1,0.5,1.0], 'alpha':0.1, 'fn_cost':1.0}
# Fix the design before looking at test outcomes.
# alpha controls the cell-frequency bands under the iid working model.
FILES = ['DEMO_L','DIQ_L','GHB_L','BMX_L','BPXO_L','PAQ_L']
CORE = ['bmi','waist','sbp']
INITIAL_FEATURE = 'bmi'
EXTRA_FEATURES = ['waist','sbp']
PANEL_A = ['bmi','waist']
PANEL_B = ['sbp']


## Files and variable meanings

Use **August 2021 - August 2023 only**. The publication dates are not separate cohorts.
Download Data [XPT] beside each component; read its Doc page. No dietary files needed.

| File | Required variables | Purpose |
|---|---|---|
| DEMO_L | SEQN, RIDAGEYR, RIAGENDR, RIDEXPRG, INDFMPIR, SDMVSTRA, SDMVPSU | IDs, eligibility, audit groups, survey design |
| DIQ_L | SEQN, DIQ010, DIQ050, DIQ070 | Reported diabetes and treatment; eligibility/audit only |
| GHB_L | SEQN, LBXGH, WTPH2YR | Outcome and phlebotomy weight |
| BMX_L | SEQN, BMXBMI, BMXWAIST | BMI and waist circumference |
| BPXO_L | SEQN, BPXOSY1–3, BPXODI1–3 | Three paired BP readings |
| PAQ_L | SEQN, PAD680 | Sedentary minutes/day; audited now, optional later feature |

- `SEQN` is an ID, never a predictor. Check uniqueness **before** merging.
- `LBXGH`, diabetes/treatment variables, survey weights and design IDs are never predictors.
- `RIAGENDR` is the recorded NHANES field; preserve its coding and describe it accurately.
- Age 80 is top-coded (80+). Do not interpret it as exact age.
- `RIDEXPRG` is publicly reported only for women aged 20–44. Preserve unknown values.
- `DIQ010`: 1=yes, 2=no, 3=borderline, 7=refused, 9=unknown.
- DIQ050/DIQ070: 1=yes, 2=no, 7/9=nonresponse; questionnaire skips are not observed 'no'.
- `PAD680`: 7777/9999 are nonresponse, not minutes. Preserve a missingness flag.

Each codebook: https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2021/DataFiles/FILE.htm
(replace FILE with the file name above).


In [16]:
# # PROVIDED: read one local XPT, checking that it is not a downloaded HTML error page.
# def read_local_xpt(path):
#     path = Path(path)
#     with path.open('rb') as handle:
#         if not handle.read(80).startswith(b'HEADER RECORD'):
#             raise ValueError(f'{path.name} is not an XPT file; re-download Data [XPT].')
#     frame = pd.read_sas(path, format='xport')
#     if 'SEQN' not in frame or frame['SEQN'].isna().any() or frame['SEQN'].duplicated().any():
#         raise ValueError(f'{path.name}: participant IDs missing or duplicated.')
#     return frame



## New work: merge without accidentally deleting patients

Read the six files. Keep DEMO as the left-hand table and left-join each other file
on SEQN, with `validate='one_to_one'`. Select the required columns above first;
avoid arbitrary suffixes hiding duplicate names. Keep a file-presence flag for each
component so missing files/rows can be distinguished from missing values within a row.

Return a join audit with file name, row count, unique IDs, and how many DEMO IDs match.
The merged row count and participant IDs must equal DEMO. Do not inner-join all six
files: that would silently turn availability into an eligibility criterion.


In [17]:
# New NHANES function: implement using the instructions above.
# - Use read_local_xpt; do not overwrite raw files.
# - Select only documented variables and preserve file-presence flags.
# Load six XPT tables and produce an audited one-row-per-participant join.
# Inputs:
#   data_dir: folder containing FILE.xpt for each of the six components.
# Outputs:
#   merged: one DataFrame, left-joined onto DEMO by SEQN.
#   join_audit: one DataFrame documenting each input and join.
#def load_and_merge_nhanes(data_dir):
# # Your code here


# Teammate #2 — Define the outcome, derive features, and audit the cohort

**New work:** implement one cohort-building function. Use the helper below for derived
measurements. Keep an auditable record of exclusions; never silently drop all missing rows.

Primary eligibility, applied in order:

1. Age >=20.
2. `DIQ010 == 2` (reports no diagnosed diabetes). Keep code 3 out of the primary cohort;
   an explicitly labeled later sensitivity can include it.
3. Exclude affirmative insulin/pill use (`DIQ050 == 1` or `DIQ070 == 1`). Missing/skipped
   treatment answers stay unknown; do not change them to 'no'.
4. Exclude known pregnancy (`RIDEXPRG == 1`). Retain other unknowns with a flag.
5. For labeled analyses, require observed positive LBXGH and positive WTPH2YR.
6. Set `target = 1` when LBXGH >=5.7, otherwise 0. Never turn missing LBXGH into a negative label.

Return the labeled eligible cohort **before** dropping missing predictors. Later the
provided runner selects participants with all three core features available so the
evaluator can reveal real values. Quantify who was excluded and retain their
availability information.

Sensitivity to report: among women 20–44, how many have unknown pregnancy status
(missing or code 3)? Show cohort sizes when those unknowns are excluded. Do not
exclude everyone whose pregnancy field is missing: most are outside its public scope.

Your audit needs, after each step: remaining n, number with HbA1c observed, and
positive-outcome count where defined. Also retain missingness/nonresponse counts.


In [ ]:
# PROVIDED: conservative, declared feature construction; preserve missing values.
def derive_nhanes_features(merged):
    d = merged.copy()
    d['bmi'] = pd.to_numeric(d['BMXBMI'], errors='coerce').where(lambda s: s > 0)
    d['waist'] = pd.to_numeric(d['BMXWAIST'], errors='coerce').where(lambda s: s > 0)
    sy = d[[f'BPXOSY{i}' for i in (1,2,3)]].apply(pd.to_numeric, errors='coerce')
    di = d[[f'BPXODI{i}' for i in (1,2,3)]].apply(pd.to_numeric, errors='coerce')
    sy.columns = di.columns = [1,2,3]
    paired = sy.gt(0) & di.gt(0)
    d['bp_pair_count'] = paired.sum(axis=1)
    d['sbp'] = sy.where(paired).mean(axis=1).where(d['bp_pair_count'] >= 2)
    d['dbp'] = di.where(paired).mean(axis=1).where(d['bp_pair_count'] >= 2)
    # Audit inconsistent readings rather than inventing undocumented exclusion cutoffs.
    d['bp_inconsistent'] = (sy.le(di) & paired).any(axis=1)
    pad = pd.to_numeric(d['PAD680'], errors='coerce')
    d['sedentary_nonresponse'] = pad.isin([7777,9999])
    d['sedentary_minutes'] = pad.where(pad.between(0,1439))
    d['pregnancy_unknown_20_44'] = (
        d['RIAGENDR'].eq(2) & d['RIDAGEYR'].between(20,44)
        & ~d['RIDEXPRG'].isin([1,2]))
    return d


In [ ]:
# New NHANES function: implement using the instructions above.
# - Call derive_nhanes_features; preserve IDs, raw outcome, audit variables and weights.
# - Follow the explicit eligibility order above and record counts after each step.
# - Do not require PAQ or all predictors to be observed at this stage.
# Apply declared eligibility rules and construct an outcome without leakage.
# Inputs:
#   merged: output of load_and_merge_nhanes.
# Outputs:
#   eligible: labeled eligible participants with target, bmi, waist, sbp, dbp,
#   SEQN, LBXGH, WTPH2YR, RIDAGEYR, RIAGENDR, INDFMPIR, RIDEXPRG,
#   SDMVSTRA, SDMVPSU, pregnancy_unknown_20_44, and measurement audit flags.
#   cohort_audit: sequential exclusion/count table.
def build_nhanes_cohort(merged):
    ...  # Your code here


## Reuse: count actual availability before artificial masking

Copy the function below unchanged. Count coverage on the eligible cohort before
complete-case selection, then on the evaluable cohort. Missingness in NHANES itself
is not assumed random; simulated panel assignment later is random by construction.


In [ ]:
# This function is from the previous prototype notebook. Copy-paste it over,
# then adjust/account for the following:
# - Copy unchanged.
# - Pass only measurement columns, not IDs, labels, or weights.
# Count individual-feature and pairwise availability.
# Inputs:
#   masked: feature DataFrame, possibly containing NaN.
# Outputs:
#   Square DataFrame of joint-observation counts, labeled by feature.
def joint_coverage(masked):
    ...  # Your code here


# Teammate #3 — Port the WDBC learners (small changes only)

Use generic NHANES names: **bmi, waist, sbp**. BMI replaces radius as the initial
measurement. The two extra choices are waist and SBP. A blood-pressure action
actually collects systolic/diastolic values together; the exact small policy uses
only SBP as its BP summary. Never charge separately for systolic and diastolic.

Copy functions into the following fields. Do not copy the old `run_split`: its
hard-coded WDBC features and nine strategies are replaced by provided runners.
You do not need a new plotting function before running the experiment.


In [ ]:
# This function is from the previous prototype notebook. Copy-paste it over,
# then adjust/account for the following:
# - Copy your WDBC function; it should already use X.columns generically.
# - Use only training medians; preserve patient row-position alignment.
# - This function is NOT the incomplete-data preprocessing: the research engine
# - uses separate calibration rows and a NaN-preserving transform.
# Split and binarize complete data for the complete-data baseline only.
# Inputs:
#   X: complete feature DataFrame in CORE order.
#   y: aligned 1D integer labels.
#   seed: reproducible split seed.
# Outputs:
#   Dictionary: train_ids, val_ids, test_ids, thresholds, z_train, z_val, z_test.
def prepare_split(X, y, seed):
    ...  # Your code here


In [ ]:
# This function is from the previous prototype notebook. Copy-paste it over,
# then adjust/account for the following:
# - Copy unchanged if ties return the integer 1, not [0,1].
# - Empty input returns fallback.
# Return a single majority diagnosis label.
# Inputs:
#   labels: 1D binary array; fallback: 0 or 1.
# Outputs:
#   One integer, 0 or 1.
def majority(labels, fallback):
    ...  # Your code here


In [ ]:
# This function is from the previous prototype notebook. Copy-paste it over,
# then adjust/account for the following:
# - Copy unchanged; use supplied column names.
# - Use the same empty-group/tie rules as majority.
# Fit a majority-label lookup reference.
# Inputs:
#   z_train: complete binary training DataFrame.
#   y_train: aligned labels; columns: feature-name list.
# Outputs:
#   Your existing lookup model, accepted by predict_lookup.
def fit_lookup(z_train, y_train, columns):
    ...  # Your code here


In [ ]:
# This function is from the previous prototype notebook. Copy-paste it over,
# then adjust/account for the following:
# - Copy unchanged.
# - Return a 1D NumPy array in the input row order.
# Apply a lookup model without using other feature columns.
# Inputs:
#   model: fit_lookup output; z: complete binary feature DataFrame.
# Outputs:
#   Predicted 0/1 labels.
def predict_lookup(model, z):
    ...  # Your code here


In [ ]:
# This function is from the previous prototype notebook. Copy-paste it over,
# then adjust/account for the following:
# - Replace hard-coded mean radius with bmi.
# - Keep choices keys at_or_below and above; values are waist or sbp.
# - Return choices and leaves keyed by (bmi_bin, acquired_bin); counts may also be returned.
# - This is the complete-data baseline, not the robust learner.
# Train one complete-data candidate tree.
# Inputs:
#   z_train: complete binary features; y_train: aligned labels.
#   choices: feature-name dictionary.
# Outputs:
#   A model dictionary with choices, leaves and training counts.
def fit_policy(z_train, y_train, choices):
    ...  # Your code here


In [ ]:
# This function is from the previous prototype notebook. Copy-paste it over,
# then adjust/account for the following:
# - Replace hard-coded mean radius with bmi.
# - Only access bmi and that row's chosen extra feature.
# - Do not require counts: robust models contain choices and leaves only.
# - Return chosen feature names as well as labels so paths can be audited.
# Follow the tree using BMI and exactly one chosen extra feature.
# Inputs:
#   model: dictionary containing choices and leaves.
#   z: complete binary features at evaluation time.
# Outputs:
#   Two arrays: predicted labels and chosen feature names, aligned to z.
def predict_policy(model, z):
    ...  # Your code here


In [ ]:
# This function is from the previous prototype notebook. Copy-paste it over,
# then adjust/account for the following:
# - Copy unchanged; target=1 now means elevated measured HbA1c.
# - Convert inputs to aligned 1D NumPy arrays; handle zero denominators with NaN.
# - Report error, false-negative/positive rates and counts, not AUPRC from hard labels.
# Measure held-out prediction errors.
# Inputs:
#   truth: observed target labels; pred: predicted labels.
# Outputs:
#   Dictionary: error_rate, false_negative_rate, false_positive_rate,
#   wrong_predictions, false_negatives, n_test, n_malignant_test.
#   Keep n_malignant_test as a compatibility key for now; it counts elevated-HbA1c cases,
#   not malignancies. Rename it in final paper outputs.
def metrics(truth, pred):
    ...  # Your code here


# Teammate #4 — Port the masking helpers and run the experiment

**Important:** the historical panels are **(BMI, waist)** and **(SBP)**, each with
the outcome. At zero overlap, we do not observe BMI and SBP together in a training
record. That makes some acquisition/prediction paths uncertain.

Do NOT instead include BMI in both panels for this small one-extra-action model:
if both BMI+waist+outcome and BMI+SBP+outcome are well observed, the risks of all these
one-extra-action policies are already determined by those panels. Our intended
relationship uncertainty would then be absent apart from sampling error.

BMI is available when making a prediction. The masking experiment varies its
availability in historical training records.


In [ ]:
# This function is from the previous prototype notebook. Copy-paste it over,
# then adjust/account for the following:
# - Set PANEL_A=[bmi,waist] and PANEL_B=[sbp] as configured above.
# - Replace hard-coded positions if present; support these three columns.
# - Reject overlap outside [0,1] and absent expected columns before get_indexer.
# - Use a floating copy so hidden values remain NaN; do not alter the original.
# - The function receives features only; outcomes are never hidden.
# Create random historical panels with controlled joint coverage.
# Inputs:
#   raw: CORE feature DataFrame; overlap_fraction: fraction keeping both panels.
#   seed: random panel-assignment seed.
# Outputs:
#   A same-shape DataFrame with panel values hidden as NaN.
def mask_panels(raw, overlap_fraction, seed):
    ...  # Your code here


In [ ]:
# This function is from the previous prototype notebook. Copy-paste it over,
# then adjust/account for the following:
# - Copy unchanged; subtract each column's best error.
# - A finite scenario table is only as comprehensive as its columns.
# - The supplied LP engine searches the full declared uncertainty set separately.
# Calculate regret for a supplied set of possible relationships.
# Inputs:
#   error_table: rows=strategies, columns=possible relationships, values=expected loss.
# Outputs:
#   Regret table, each strategy's largest regret, and names of all tied minimizers.
def calculate_regret(error_table):
    ...  # Your code here


## PROVIDED — What the research engine does

You do not need to implement the mathematics below. Read this description and run
the synthetic controls before the real-data grid.

1. **Split first.** Training, validation and test patients remain separate. Within
   training, separate calibration patients set median thresholds from randomly
   fragmented records only. Their labels/joint values never train the acquisition model.
2. **Hide training panels.** Vary record count and overlap separately. Thresholds and
   test patients stay fixed across overlap/sample settings within a seed.
3. **Describe uncertainty.** A distribution has 16 possible states: three binary
   measurements and a binary outcome. Constrain its observable panel frequencies
   using finite-sample bands. Never declare an unobserved state impossible just
   because it did not occur. Stop and report an infeasible set; do not repair it with test data.
4. **Learn acquisition AND final predictions.** Search all 36 distinct allowable
   predictors, rather than freezing endpoint labels learned from hidden complete data.
5. **Compare methods fairly:** a point-completion assumption, minimax total risk,
   minimax regret, and a well-observed single-panel strategy. They share the same
   inputs, candidate class and one-extra-action budget.
6. **Evaluate on real held-out values.** The learner receives masked calibration/fit
   records only. The evaluator reveals one selected extra feature for each test patient.

The main research grid fixes its hyperparameters in CONFIG and reserves the validation
partition without using its complete joint values to select a policy. That avoids
quietly resolving the missing-relationship problem through fully observed validation.
The complete-data WDBC-style bridge has its own ordinary validation selection and
split convention; treat it as a separate diagnostic baseline, not a paired comparison
with the research grid. All methods within the research grid share the same splits.

For minimax regret, the candidate and comparator must share **one entire possible
distribution**. The engine maximizes their loss difference over that set, then
selects the candidate with the smallest such maximum. The comparator has the same
budget and policy class. All optimization is finite and exact up to LP tolerances.

Frequency bands use an iid concentration formula with the configured alpha.
Separate calibration rows set thresholds; the remaining training rows estimate
panel frequencies.

Primary training loss counts false positives and false negatives equally. Weighted
error/FNR/FPR are secondary survey-weight sensitivities using WTPH2YR; they are not
the objective of the unweighted optimizer. A predeclared fn_cost=2 run is an optional
loss-sensitivity experiment.


In [ ]:
"""Supplied experiment engine for the starter notebook.

Optimize over a finite deterministic policy class using linear programming.
"""
from itertools import product
import hashlib
import json
import platform
import time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import linprog
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score

CORE = ['bmi', 'waist', 'sbp']
STATES = np.array(list(product((0, 1), repeat=4)), dtype=int)  # BMI, waist, SBP, label

def enumerate_policies():
    """64 descriptions / 36 distinct predictors; every rule reads BMI + one extra."""
    models, predictions, seen = [], [], set()
    for low, high in product(['waist', 'sbp'], repeat=2):
        for labels in product((0, 1), repeat=4):
            choices = {'at_or_below': low, 'above': high}
            leaves = dict(zip(product((0, 1), repeat=2), labels))
            pred = []
            for b, w, s, _ in STATES:
                j = low if b == 0 else high
                pred.append(leaves[(b, w if j == 'waist' else s)])
            key = tuple(pred)
            if key in seen:
                continue
            seen.add(key)
            models.append({'choices': choices.copy(), 'leaves': leaves.copy()})
            predictions.append(pred)
    return models, np.asarray(predictions, dtype=int)

POLICIES, STATE_PREDICTIONS = enumerate_policies()

def state_codes(z, labels):
    a = np.column_stack([z.loc[:, CORE].to_numpy(), labels])
    if not np.isfinite(a).all() or not np.isin(a, [0, 1]).all():
        raise ValueError('Evaluator requires complete binary features and labels.')
    return a.astype(int) @ np.array([8, 4, 2, 1])

def loss_matrix(fn_cost=1.0):
    truth = STATES[:, 3]
    return ((STATE_PREDICTIONS != truth) * np.where(truth == 1, fn_cost, 1)).astype(float)

def build_uncertainty(masked, labels, alpha=0.1):
    """Simultaneous cell-frequency bands; fixed preprocessing; random panel assignment.

    Use Hoeffding bands with the configured alpha under an iid working model.
    """
    if not 0 < alpha < 1:
        raise ValueError('alpha must lie strictly between 0 and 1')
    arr = np.column_stack([masked.loc[:, CORE].to_numpy(), labels])
    if not np.isin(arr[np.isfinite(arr)], [0, 1]).all():
        raise ValueError('Observed features and all labels must be binary.')
    if not np.isfinite(arr[:, 3]).all():
        raise ValueError('Outcome labels must be observed.')
    groups = [(3,), (0, 1, 3), (2, 3), (0, 1, 2, 3)]
    total_cells = sum(2 ** len(g) for g in groups)
    rows, rhs, audit = [], [], []
    for group in groups:
        keep = np.isfinite(arr[:, group]).all(axis=1)
        n = int(keep.sum())
        if n == 0:
            audit.append({'group': str(group), 'n': 0, 'radius': np.nan})
            continue  # Absence of data imposes no zero-probability constraints.
        radius = np.sqrt(np.log(2 * total_cells / alpha) / (2 * n))
        audit.append({'group': str(group), 'n': n, 'radius': float(radius)})
        for values in product((0, 1), repeat=len(group)):
            indicator = np.all(STATES[:, group] == np.array(values), axis=1).astype(float)
            q = np.mean(np.all(arr[keep][:, group] == np.array(values), axis=1))
            lo, hi = max(0, q - radius), min(1, q + radius)
            rows.extend([indicator, -indicator])
            rhs.extend([hi, -lo])
    U = {'A': np.array(rows), 'b': np.array(rhs), 'audit': pd.DataFrame(audit)}
    # Feasibility check; never silently widen a set using hidden/test values.
    U['feasible'] = solve_distribution(U, np.zeros(16))[1]
    return U

def solve_distribution(U, objective):
    result = linprog(np.asarray(objective, float), A_ub=U['A'], b_ub=U['b'],
                     A_eq=np.ones((1, 16)), b_eq=[1.0], bounds=(0, None), method='highs')
    if not result.success:
        raise RuntimeError(f'Uncertainty LP failed ({result.status}): {result.message}')
    p = result.x
    if (p.min() < -1e-7 or abs(p.sum() - 1) > 1e-7
            or np.max(U['A'] @ p - U['b']) > 1e-7):
        raise RuntimeError('LP witness violates declared constraints beyond tolerance.')
    return float(result.fun), p

def point_completion(masked, labels, U):
    """One explicit assumption: (BMI, waist) independent of SBP conditional on label.

    Estimate both panel conditionals from observed values only; smooth counts.
    Project this completion to U in L1 distance.
    """
    arr = np.column_stack([masked.loc[:, CORE].to_numpy(), labels])
    p0 = np.zeros(16)
    for yy in [0, 1]:
        label_mask = arr[:, 3] == yy
        prior = (label_mask.sum() + 1) / (len(labels) + 2)
        a = np.ones((2, 2))
        for row in arr[label_mask & np.isfinite(arr[:, :2]).all(axis=1)]:
            a[int(row[0]), int(row[1])] += 1
        a /= a.sum()
        b = np.ones(2)
        for row in arr[label_mask & np.isfinite(arr[:, 2])]:
            b[int(row[2])] += 1
        b /= b.sum()
        for i, (x, w, s, y0) in enumerate(STATES):
            if y0 == yy:
                p0[i] = prior * a[x, w] * b[s]
    # Minimize sum t_i subject to |p_i - p0_i| <= t_i and p in U.
    A = np.vstack([np.column_stack([U['A'], np.zeros_like(U['A'])]),
                   np.column_stack([np.eye(16), -np.eye(16)]),
                   np.column_stack([-np.eye(16), -np.eye(16)])])
    b = np.r_[U['b'], p0, -p0]
    result = linprog(np.r_[np.zeros(16), np.ones(16)], A_ub=A, b_ub=b,
                     A_eq=np.r_[np.ones(16), np.zeros(16)][None, :], b_eq=[1],
                     bounds=(0, None), method='highs')
    if not result.success:
        raise RuntimeError('Point-completion projection failed: ' + result.message)
    return result.x[:16]

def optimize_finite(U, point, fn_cost=1.0):
    """For each policy i compute max_P max_j E_P[loss_i - loss_j].

    BOTH policies share ONE distribution P for every state/history. Candidate and
    comparator range over the same 36 distinct predictors. All leaf labels are
    optimized: they are not frozen to complete-data majority predictions.
    """
    started = time.perf_counter()
    L = loss_matrix(fn_cost)
    cache = {}
    def worst(v):
        key = tuple(v)
        if key not in cache:
            value, witness = solve_distribution(U, -v)
            cache[key] = (-value, witness)
        return cache[key]
    n = len(POLICIES)
    pair = np.empty((n, n))
    for i in range(n):
        for j in range(n):
            pair[i, j] = worst(L[i] - L[j])[0]
    regret = np.maximum(0, pair.max(axis=1))
    worst_risk = np.array([worst(v)[0] for v in L])
    point_risk = L @ point
    selected = {'point_completion': int(np.argmin(point_risk)),
                'minimax_risk': int(np.argmin(worst_risk)),
                'minimax_regret': int(np.argmin(regret))}
    # A strategy supported by panel A; select leaf labels under observed-panel completion.
    waist_only = [i for i, m in enumerate(POLICIES)
                  if set(m['choices'].values()) == {'waist'}]
    selected['single_panel_waist'] = min(waist_only, key=lambda i: (point_risk[i], i))
    i = selected['minimax_regret']
    j = int(np.argmax(pair[i]))
    value, witness = worst(L[i] - L[j])
    # Independently bounded risks are generally looser than shared-P contrasts.
    min_risks = np.array([-worst(-v)[0] for v in L])
    independent_upper = worst_risk - min_risks.min()
    scores = pd.DataFrame({'policy_id': range(n), 'point_risk': point_risk,
        'worst_risk': worst_risk, 'worst_regret': regret,
        'independent_difference_upper': independent_upper})
    scores['tied_minimax_regret'] = np.isclose(regret, regret.min(), atol=1e-7, rtol=0)
    scores['tied_minimax_risk'] = np.isclose(worst_risk, worst_risk.min(), atol=1e-7, rtol=0)
    scores['tied_point_optimum'] = np.isclose(point_risk, point_risk.min(), atol=1e-7, rtol=0)
    return {'selected': selected, 'scores': scores, 'pairwise': pair,
            'witness': witness, 'witness_comparator': j, 'witness_regret': value,
            'seconds': time.perf_counter() - started, 'lp_calls': len(cache),
            'losses': L}

def binary_transform(raw, thresholds):
    """Keep NaN as NaN. A missing value is never an observed low value."""
    z = raw.loc[:, CORE].gt(thresholds, axis='columns').astype(float)
    return z.where(raw.loc[:, CORE].notna())

def split_research_ids(labels, seed):
    ids = np.arange(len(labels))
    trainval, test = train_test_split(ids, test_size=.2, stratify=labels, random_state=seed)
    train, val = train_test_split(trainval, test_size=.25, stratify=labels[trainval], random_state=seed+50)
    # Disjoint calibration rows set thresholds; they do not enter U or model training.
    fit, calibration = train_test_split(train, test_size=.2, stratify=labels[train], random_state=seed+100)
    return {'fit': fit, 'calibration': calibration, 'validation': val, 'test': test}

def weighted_metrics(truth, pred, weights):
    truth, pred, weights = map(np.asarray, [truth, pred, weights])
    if len(truth) != len(pred) or len(truth) != len(weights) or (weights <= 0).any():
        raise ValueError('Need aligned labels, predictions and positive weights.')
    def mean(v, use):
        return float(np.average(v[use], weights=weights[use])) if use.any() else np.nan
    return {'weighted_error': mean(truth != pred, np.ones(len(truth), dtype=bool)),
            'weighted_fnr': mean(pred == 0, truth == 1),
            'weighted_fpr': mean(pred == 1, truth == 0)}

def working_psu_interval(truth, pred, meta, seed=0, repetitions=200):
    """Resample observed PSUs within strata and summarize frozen-model prediction error.
    """
    rng = np.random.default_rng(seed)
    meta = meta.reset_index(drop=True)
    blocks = []
    for _, stratum in meta.groupby('SDMVSTRA', dropna=False):
        psus = [g.index.to_numpy() for _, g in stratum.groupby('SDMVPSU', dropna=False)]
        if len(psus) < 2:
            return {'weighted_error_lo': np.nan, 'weighted_error_hi': np.nan,
                    'interval_note': 'Not estimated: a test stratum has fewer than two observed PSUs.'}
        blocks.append(psus)
    estimates = []
    for _ in range(repetitions):
        ids = np.concatenate([np.concatenate([b[j] for j in rng.integers(len(b), size=len(b))])
                              for b in blocks])
        estimates.append(weighted_metrics(truth[ids], pred[ids], meta['WTPH2YR'].to_numpy()[ids])['weighted_error'])
    lo, hi = np.quantile(estimates, [.025, .975])
    return {'weighted_error_lo': float(lo), 'weighted_error_hi': float(hi),
            'interval_note': 'Exploratory within-stratum PSU bootstrap; frozen model, selected cohort.'}

def run_complete_baseline(cohort, seeds, prepare_split, fit_lookup, predict_lookup,
                          fit_policy, predict_policy, metrics):
    """Reuse WDBC learners; complete-data assistance is explicitly labeled."""
    X = cohort[CORE].reset_index(drop=True)
    y = cohort['target'].to_numpy(dtype=int)
    rows, traces = [], []
    for seed in seeds:
        split = prepare_split(X, y, seed)
        tr, va, te = [split[k] for k in ['train_ids', 'val_ids', 'test_ids']]
        ztr, zv, zte = [split[k] for k in ['z_train', 'z_val', 'z_test']]
        models = []
        for low, high in product(['waist', 'sbp'], repeat=2):
            m = fit_policy(ztr, y[tr], {'at_or_below': low, 'above': high})
            pv, _ = predict_policy(m, zv)
            models.append((float(np.mean(pv != y[va])), low, high, m))
        best = min(models, key=lambda t: t[:3])
        best_fixed = min([t for t in models if t[1] == t[2]], key=lambda t: t[:3])
        for name, columns, count in [('bmi_only', ['bmi'], 0), ('all_core', CORE, 2)]:
            m = fit_lookup(ztr, y[tr], columns)
            pred = predict_lookup(m, zte)
            rows.append({'seed': seed, 'method': name, 'extra_actions': count,
                         'training_information': 'complete', **metrics(y[te], pred)})
        for name, candidate in [('best_fixed', best_fixed), ('adaptive', best)]:
            pred, chosen = predict_policy(candidate[3], zte)
            rows.append({'seed': seed, 'method': name, 'extra_actions': 1,
                         'training_information': 'complete', **metrics(y[te], pred)})
            for j, pos in enumerate(te):
                traces.append({'seed': seed, 'method': name, 'SEQN': int(cohort.iloc[pos]['SEQN']),
                    'chosen': chosen[j], 'prediction': int(pred[j]), 'truth': int(y[pos])})
    return pd.DataFrame(rows), pd.DataFrame(traces)

def run_research_grid(cohort, config, mask_panels, predict_policy, metrics, calculate_regret):
    """Learners see masked calibration/fit tables only; evaluator retains hidden truth.

    The predeclared unweighted finite-cohort loss is the training objective. Survey-
    weighted errors are sensitivity summaries, not the objective optimized by U.
    """
    raw = cohort[CORE].reset_index(drop=True)
    y = cohort['target'].to_numpy(dtype=int)
    outputs = {k: [] for k in ['results', 'coverage', 'policy_scores', 'witnesses',
                              'traces', 'subgroups', 'policy_comparisons', 'scenario_audit',
                              'splits', 'failures']}
    for seed in config['seeds']:
        split = split_research_ids(y, seed)
        for part, ids in split.items():
            outputs['splits'].extend({'seed': seed, 'partition': part, 'SEQN': int(cohort.iloc[i]['SEQN'])}
                                     for i in ids)
        calibration = mask_panels(raw.iloc[split['calibration']], 0.0, seed+200)
        thresholds = calibration.median()
        if thresholds.isna().any():
            raise ValueError('Calibration panel lacks an observed value for a core feature.')
        ztest = binary_transform(raw.iloc[split['test']], thresholds).reset_index(drop=True)
        ytest = y[split['test']]
        meta = cohort.iloc[split['test']].reset_index(drop=True)
        test_codes = state_codes(ztest, ytest)
        # Evaluator-only: same-class empirical best on this test sample. Optimistic
        # descriptive reference calculated from the test sample.
        test_errors = (STATE_PREDICTIONS[:, test_codes] != ytest).mean(axis=1)
        for fraction in config['sample_fractions']:
            if fraction == 1:
                fit_ids = split['fit']
            else:
                fit_ids, _ = train_test_split(split['fit'], train_size=fraction,
                    stratify=y[split['fit']], random_state=seed+300)
            for overlap in config['overlaps']:
                tag = {'seed': seed, 'sample_fraction': fraction, 'overlap': overlap,
                       'n_fit': len(fit_ids)}
                hidden = mask_panels(raw.iloc[fit_ids], overlap, seed+400)
                zfit = binary_transform(hidden, thresholds).reset_index(drop=True)
                yfit = y[fit_ids]
                label_data = pd.concat([zfit, pd.Series(yfit, name='label')], axis=1)
                outputs['coverage'].append({**tag, 'n_complete': int(zfit.notna().all(axis=1).sum()),
                    **{f'n_{c}': int(zfit[c].notna().sum()) for c in CORE},
                    'n_bmi_sbp': int(zfit[['bmi','sbp']].notna().all(axis=1).sum())})
                try:
                    U = build_uncertainty(zfit, yfit, alpha=config['alpha'])
                    point = point_completion(zfit, yfit, U)
                    opt = optimize_finite(U, point, fn_cost=config['fn_cost'])
                except RuntimeError as e:
                    outputs['failures'].append({**tag, 'reason': str(e)})
                    continue
                scores = opt['scores'].copy()
                scores['test_error_evaluator_only'] = test_errors
                for key, value in tag.items(): scores[key] = value
                outputs['policy_scores'].extend(scores.to_dict('records'))
                # Learned methods and matched-class fixed baseline are evaluated identically.
                for name, pid in opt['selected'].items():
                    model = POLICIES[pid]
                    pred, chosen = predict_policy(model, ztest)
                    expected = STATE_PREDICTIONS[pid, test_codes]
                    if not np.array_equal(pred, expected):
                        raise AssertionError('Copied predictor disagrees with finite policy representation.')
                    weights = meta['WTPH2YR'].to_numpy()
                    score = opt['scores'].iloc[pid]
                    outputs['results'].append({**tag, 'method': name, 'policy_id': pid,
                        'extra_actions': 1, 'objective_fn_cost': config['fn_cost'],
                        'set_worst_regret': score['worst_regret'], 'set_worst_risk': score['worst_risk'],
                        'empirical_test_class_excess_error': float(test_errors[pid]-test_errors.min()),
                        'seconds': opt['seconds'], 'lp_calls': opt['lp_calls'],
                        'threshold_bmi': thresholds['bmi'], 'threshold_waist': thresholds['waist'],
                        'threshold_sbp': thresholds['sbp'], **metrics(ytest, pred),
                        **weighted_metrics(ytest, pred, weights)})
                    for row in range(len(meta)):
                        outputs['traces'].append({**tag, 'method': name,
                            'SEQN': int(meta.iloc[row]['SEQN']), 'initial_bin': int(ztest.iloc[row]['bmi']),
                            'chosen': chosen[row], 'acquired_bin': int(ztest.iloc[row][chosen[row]]),
                            'prediction': int(pred[row]), 'truth': int(ytest[row])})
                    # Prespecified descriptive subgroup summaries; suppress unstable small cells.
                    age_groups = pd.cut(meta['RIDAGEYR'], [19,39,59,np.inf], labels=['20-39','40-59','60+'])
                    groups = {'age': age_groups, 'recorded_sex': meta['RIAGENDR'],
                              'income_poverty': pd.cut(meta['INDFMPIR'], [-np.inf,1.3,np.inf], labels=['<=1.3','>1.3'])}
                    for variable, series in groups.items():
                        for group in series.dropna().unique():
                            take = (series == group).to_numpy()
                            n, positive = int(take.sum()), int(ytest[take].sum())
                            small = n < 30 or positive < 10 or n-positive < 10
                            vals = metrics(ytest[take], pred[take]) if not small else {}
                            outputs['subgroups'].append({**tag,'method':name,'variable':variable,
                                'group':str(group),'n':n,'n_positive':positive,'suppressed':small,**vals})
                # Store a replayable worst-case distribution for the selected minimax-regret policy.
                pid = opt['selected']['minimax_regret']; comp = opt['witness_comparator']
                scenario_errors = pd.DataFrame(opt['losses'] @ np.column_stack([point, opt['witness']]),
                    columns=['point_completion', 'regret_witness'], index=pd.Index(range(len(POLICIES)), name='policy_id'))
                scenario_regret, finite_worst, _ = calculate_regret(scenario_errors)
                if abs(float(scenario_regret.loc[pid, 'regret_witness']) - opt['scores'].iloc[pid]['worst_regret']) > 1e-6:
                    raise AssertionError('Witness does not replay the selected policy regret.')
                for policy_id in scenario_errors.index:
                    for scenario in scenario_errors.columns:
                        outputs['scenario_audit'].append({**tag, 'policy_id': int(policy_id),
                            'scenario':scenario, 'expected_loss':float(scenario_errors.loc[policy_id,scenario]),
                            'scenario_regret':float(scenario_regret.loc[policy_id,scenario]),
                            'note':'Two diagnostic scenarios only; full worst regret is in policy_scores.'})
                for state, mass in zip(STATES, opt['witness']):
                    outputs['witnesses'].append({**tag, **dict(zip(CORE+['label'], state.tolist())),
                        'mass': float(mass), 'selected_policy': pid, 'comparator_policy': comp,
                        'attained_regret': opt['witness_regret']})
                # Compare selected fixed policies under the shared uncertainty set.
                pids = list(dict.fromkeys(opt['selected'].values()))
                for a in pids:
                    for b in pids:
                        upper = opt['pairwise'][a,b]; lower = -opt['pairwise'][b,a]
                        outputs['policy_comparisons'].append({**tag,'policy_a':a,'policy_b':b,
                            'loss_difference_lower':lower,'loss_difference_upper':upper,
                            'ordering_resolved': bool(upper < -1e-7 or lower > 1e-7)})
                print(f'seed={seed} fraction={fraction} overlap={overlap}: {opt["lp_calls"]} LPs, {opt["seconds"]:.1f}s')
    return {key: pd.DataFrame(rows) for key, rows in outputs.items()}

def clinical_references(cohort, seeds):
    """Fit continuous logistic comparison models with demographic context.
    """
    rows=[]
    y=cohort['target'].to_numpy(dtype=int)
    complete = cohort[['RIDAGEYR','RIAGENDR','bmi','waist','sbp']].notna().all(axis=1)
    if not complete.all():
        raise ValueError('Reference models must use exactly the same evaluable cohort.')
    for seed in seeds:
        ids=split_research_ids(y,seed); tr=np.r_[ids['fit'],ids['calibration']]; te=ids['test']
        for name, columns in {'age_sex_bmi':['RIDAGEYR','RIAGENDR','bmi'],
                              'age_sex_bmi_waist':['RIDAGEYR','RIAGENDR','bmi','waist'],
                              'age_sex_bmi_sbp':['RIDAGEYR','RIAGENDR','bmi','sbp'],
                              'collect_all_core':['RIDAGEYR','RIAGENDR','bmi','waist','sbp']}.items():
            model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000))
            model.fit(cohort.iloc[tr][columns],y[tr])
            prob=model.predict_proba(cohort.iloc[te][columns])[:,1]
            rows.append({'seed':seed,'reference':name,'training_information':'complete',
                'model_class':'continuous logistic; extra demographic context',
                'error':float(np.mean((prob>=.5)!=y[te])),
                'auprc':float(average_precision_score(y[te],prob)),
                'auroc':float(roc_auc_score(y[te],prob)),
                'weighted_error':weighted_metrics(y[te],prob>=.5,cohort.iloc[te]['WTPH2YR'].to_numpy())['weighted_error']})
    return pd.DataFrame(rows)

def export_bundle(outputs, path, config, source_paths):
    path=Path(path); path.mkdir(parents=True,exist_ok=True)
    for key, table in outputs.items(): table.to_csv(path/f'{key}.csv',index=False)
    files={str(p):hashlib.sha256(Path(p).read_bytes()).hexdigest() for p in source_paths}
    manifest={'config':config,'source_sha256':files,'python':platform.python_version(),
              'numpy':np.__version__,'pandas':pd.__version__,
              'clinical_target':'observed HbA1c >= 5.7%',
              'policy_class':'BMI initially known, one additional waist or BP action, binary inputs',
              'uncertainty_note':'iid cell-frequency bands with disjoint threshold calibration',
              'policy_library':[{'choices':m['choices'],'leaves':{str(k):v for k,v in m['leaves'].items()}} for m in POLICIES]}
    (path/'manifest.json').write_text(json.dumps(manifest,indent=2))


## PROVIDED — Check your copied functions before using NHANES

Run this after completing the copy fields. It uses invented binary data so you
can debug integration before finishing table preparation.


In [ ]:
def check_function_ports():
    z = pd.DataFrame({'bmi':[0,0,1,1,0,1], 'waist':[0,1,0,1,1,0], 'sbp':[1,0,1,0,0,1]})
    y = np.array([0,1,1,0,1,1])
    assert majority(np.array([0,1]), 0) == 1, 'Tie must return integer 1.'
    assert majority(np.array([],dtype=int), 0) == 0
    for feature in EXTRA_FEATURES:
        model = fit_policy(z,y,{'at_or_below':feature,'above':feature})
        pred, chosen = predict_policy(model,z)
        ref = predict_lookup(fit_lookup(z,y,['bmi',feature]), z)
        np.testing.assert_array_equal(pred,ref)
        assert set(chosen) == {feature}
    model = fit_policy(z,y,{'at_or_below':'waist','above':'sbp'})
    pred, chosen = predict_policy(model,z)
    changed = z.copy()
    for i, feature in enumerate(chosen):
        other = 'sbp' if feature == 'waist' else 'waist'
        changed.loc[i,other] = 1-changed.loc[i,other]
    pred2, chosen2 = predict_policy(model,changed)
    np.testing.assert_array_equal(pred,pred2)
    np.testing.assert_array_equal(chosen,chosen2)
    raw = z.astype(float); original=raw.copy(deep=True)
    for overlap in [0,.1,.5,1]:
        a=mask_panels(raw,overlap,7); b=mask_panels(raw,overlap,7)
        pd.testing.assert_frame_equal(a,b)
        pd.testing.assert_frame_equal(raw,original)
        assert a.notna().all(axis=1).sum() == round(overlap*len(raw))
        cov=joint_coverage(a)
        assert cov.loc['bmi','sbp'] == round(overlap*len(raw))
    missing=raw.copy();missing.loc[0,'sbp']=np.nan
    assert np.isnan(binary_transform(missing,raw.median()).loc[0,'sbp'])
    print('PASS: copied functions agree, acquisition does not peek, masks preserve input and missingness.')

# Uncomment after completing Teammates #3/#4 copy fields:
# check_function_ports()


## PROVIDED — Synthetic controls for the optimization engine

The first control creates two different complete-data relationships with identical
observed panel marginals. More disconnected records cannot distinguish them. The
second reveals the full joint distribution and verifies that attainable minimum
worst regret becomes zero. The checks use exact toy distributions.


In [ ]:
def run_synthetic_controls():
    # BMI and SBP determine the outcome by XOR; waist is independent.
    p=np.array([1/8 if yy == (b != s) else 0 for b,w,s,yy in STATES])
    q=np.array([1/8 if yy == (b == s) else 0 for b,w,s,yy in STATES])
    def exact_set(groups):
        A=[];rhs=[]
        for group in groups:
            for values in product((0,1),repeat=len(group)):
                row=np.all(STATES[:,group]==values,axis=1).astype(float)
                mass=float(row@p);A.extend([row,-row]);rhs.extend([mass,-mass])
        return {'A':np.array(A),'b':np.array(rhs),'feasible':p}
    incomplete=exact_set([(0,1,3),(2,3)])
    assert np.max(incomplete['A']@q-incomplete['b']) < 1e-10
    one=optimize_finite(incomplete,p)
    full=exact_set([(0,1,2,3)])
    two=optimize_finite(full,p)
    assert np.isclose(one['scores']['worst_regret'].min(),.5)
    assert np.isclose(two['scores']['worst_regret'].min(),0)
    display(pd.DataFrame({'control':['Disconnected exact panels','Full exact joint'],
        'minimum_worst_regret':[one['scores']['worst_regret'].min(),two['scores']['worst_regret'].min()]}))
    print('PASS: ambiguity remains with exact disconnected marginals; full observation resolves this toy.')

run_synthetic_controls()


## PROVIDED — Build the auditable cohort and run a first pass

Set RUN_REAL_DATA=True only when copy fields and two new functions are complete.
The runner exports artifacts, not just printed scores. Start with the quick grid,
then set RUN_FULL_GRID=True for 3 seeds × 2 sample sizes × 4 overlaps.

The outcome/cohort must be fixed before inspecting test performance. Do not choose
features, exclusion rules, alpha, or a preferred split based on the best test result.
Exploratory changes need a new clearly labeled evaluation plan, not a rewritten
story around the same test scores. 


In [ ]:
if RUN_REAL_DATA:
    check_function_ports()
    merged, join_audit = load_and_merge_nhanes(DATA_DIR)
    eligible, cohort_audit = build_nhanes_cohort(merged)
    required = ['SEQN','target','LBXGH','WTPH2YR','RIDAGEYR','RIAGENDR','INDFMPIR',
                'SDMVSTRA','SDMVPSU','pregnancy_unknown_20_44'] + CORE
    absent = set(required)-set(eligible.columns)
    if absent: raise ValueError(f'Cohort missing columns: {sorted(absent)}')
    assert eligible['SEQN'].is_unique
    assert eligible['LBXGH'].notna().all()
    assert eligible['WTPH2YR'].gt(0).all()
    np.testing.assert_array_equal(eligible['target'].to_numpy(),(eligible['LBXGH']>=5.7).astype(int).to_numpy())
    original_coverage = joint_coverage(eligible[CORE])
    can_evaluate = eligible[CORE].notna().all(axis=1)
    cohort = eligible.loc[can_evaluate].reset_index(drop=True)
    availability_audit = eligible[['SEQN','target','RIDAGEYR','RIAGENDR','INDFMPIR','WTPH2YR']].copy()
    availability_audit['evaluable'] = can_evaluate.to_numpy()
    for feature in CORE: availability_audit[f'observed_{feature}'] = eligible[feature].notna().to_numpy()
    print('Labeled eligible:',len(eligible),'| Evaluable:',len(cohort),'| Positive outcomes:',int(cohort['target'].sum()))
    display(join_audit);display(cohort_audit);display(original_coverage)
    display(availability_audit.groupby('evaluable',dropna=False).agg(
        n=('SEQN','size'),positive_fraction=('target','mean'),mean_age=('RIDAGEYR','mean')))
    # No automatic target switch or silent cohort relaxation if counts are small.
    if cohort['target'].nunique()!=2: raise ValueError('Need both outcome classes; inspect the cohort audit.')
    active = dict(CONFIG)
    if not RUN_FULL_GRID: active.update(seeds=[0],sample_fractions=[1.0])
    baseline, baseline_traces = run_complete_baseline(cohort,active['seeds'],prepare_split,
        fit_lookup,predict_lookup,fit_policy,predict_policy,metrics)
    research = run_research_grid(cohort,active,mask_panels,predict_policy,metrics,calculate_regret)
    research.update({'complete_baseline':baseline,'baseline_traces':baseline_traces,
        'join_audit':join_audit,'cohort_audit':cohort_audit,'availability_audit':availability_audit,
        'natural_coverage':original_coverage.rename_axis('feature').reset_index(),
        'clinical_references':clinical_references(cohort,active['seeds'])})
    export_bundle(research,OUTPUT_DIR,active,[DATA_DIR/f'{name}.xpt' for name in FILES])
    display(research['results'])
    if not research['failures'].empty: display(research['failures'])
else:
    print('Starter notebook ready. Complete the function bodies, set DATA_DIR, then enable RUN_REAL_DATA.')


## PROVIDED — Main figures and tables

Keep measured test error and set-based regret on separate axes. A smaller bound
does not automatically mean fewer mistakes in the actual held-out cohort.


In [ ]:
if RUN_REAL_DATA and not research['results'].empty:
    summary = research['results'].groupby(['sample_fraction','overlap','method']).agg(
        mean_test_error=('error_rate','mean'),split_sd=('error_rate','std'),
        mean_fnr=('false_negative_rate','mean'),mean_fpr=('false_positive_rate','mean'),
        mean_set_regret=('set_worst_regret','mean'),mean_weighted_error=('weighted_error','mean'),
        mean_empirical_class_excess=('empirical_test_class_excess_error','mean')).reset_index()
    summary.to_csv(OUTPUT_DIR/'main_summary.csv',index=False)
    display(summary)
    full=summary[summary['sample_fraction']==1.0]
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for method,g in full.groupby('method'):
        g=g.sort_values('overlap')
        axes[0].plot(g['overlap'],g['mean_test_error'],marker='o',label=method)
        axes[1].plot(g['overlap'],g['mean_set_regret'],marker='o',label=method)
    axes[0].set_ylabel('Held-out prediction error')
    axes[1].set_ylabel('Worst regret over declared uncertainty set')
    for ax in axes: ax.set_xlabel('Training joint-observation fraction');ax.legend(fontsize=8)
    fig.tight_layout();fig.savefig(OUTPUT_DIR/'coverage_vs_performance.png',dpi=160);plt.show()
    # Finite class/test-set excess uses test labels to define a descriptive reference.


## What the completed notebook must hand back

1. **Data integrity:** join audit, sequential cohort exclusions, source hashes,
   natural missingness, and evaluable-versus-excluded participant summaries.
2. **Complete-data baselines:** initial-only, best fixed, adaptive, and all-core
   references; prediction-path traces. The all-core reference uses a larger budget.
3. **Research grid:** independently varied sample size and overlap, with fixed
   calibration/test patients within seed; selected policies, error, FNR/FPR,
   set-based regret, weighted sensitivities, runtime, and failure logs.
4. **Mechanism evidence:** feasible worst-case distributions, comparator policies,
   diagnostic scenario regrets, and intervals for selected fixed-policy comparisons.
   The same distribution is used for both policies in every comparison.
5. **Subgroup summaries:** age, recorded sex, income/poverty; counts and explicit
   small-cell suppression.
6. **Practical reference:** continuous logistic models with demographic context,
   including collecting all inexpensive core inputs. These have extra information,
   a different model class and complete training data: label them accordingly.
